In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

using Plots
using Printf

In [ ]:
# Minimal Julia implementation of the `neo_2021` branch.

# Inputs:
#   Te     : electron temperature [eV]
#   ne     : electron density [m^-3]
#   Ti     : ion temperature [eV]
#   q      : safety factor
#   eps    : inverse aspect ratio
#   R      : major radius [m]
#   fT     : trapped-particle fraction
#   Zeff   : effective charge

# Output:
#   σ_spitzer : Spitzer conductivity [1/(Ω m)]
#   σ_neo     : Redl 2021 neoclassical conductivity [1/(Ω m)]

# All inputs can be scalars or arrays of the same size.

function neoclassical_conductivity(
    Te, ne, Ti, q, eps, R, fT, Zeff;
    version = :neo_2021,
)

    # Coulomb logarithms
    lnΛe = 23.5 .-
           log.(sqrt.(ne ./ 1e6) .* Te.^(-5/4)) .-
           sqrt.(1e-5 .+ (log.(Te) .- 2).^2 ./ 16)

    # Ion density.
    #
    # For the simple D + C model used in the original code:
    #     n_i = n_main + n_imp
    #
    # If only Zeff is supplied, the original function uses
    #     n_i = ne / Zeff
    ni = ne ./ Zeff

    # With only Zeff available, Zdom = 1, giving
    Zdom = 1.0

    # Average ion charge
    Zavg = ne ./ ni

    # Koh effective ion charge
    Zion = (Zdom^2 .* Zavg .* Zeff).^(1/4)

    # Ion Coulomb logarithm
    lnΛi = 30.0 .-
           log.(Zion.^3 .* sqrt.(ni) ./ Ti.^1.5)

    # Prevent pathological negative electron Coulomb logarithms
    lnΛe_min = minimum(lnΛe[lnΛe .> 0])
    lnΛe = max.(lnΛe, lnΛe_min)

    # Avoid eps = 0
    eps_nz = copy(eps)
    if any(eps .== 0)
        eps_nz[eps .== 0] .= minimum(eps[eps .!= 0])
    end

    # Electron collisionality, Eq. 18b
    νe_star = 6.921e-18 .* abs.(q) .* R .* ne .* Zeff .* lnΛe ./
              (Te.^2 .* eps_nz.^1.5)

    # Ion collisionality, Eq. 18c
    νi_star = 4.90e-18 .* abs.(q) .* R .* ni .* Zion.^4 .* lnΛi ./
              (Ti.^2 .* eps_nz.^1.5)

    # Sauter Eq. 18a
    NZ(Z) = 0.58 + 0.74 ./ (0.76 + Z)

    σ_spitzer = 1.9012e4 .* Te.^1.5 ./
                (Zeff .* NZ(Zeff) .* lnΛe)

    # Redl 2021, Eq. 18
    f33teff = fT ./ (
        1 .+
        0.25 .* (1 .- 0.7 .* fT) .* sqrt.(νe_star) .*
        (1 .+ 0.45 .* sqrt.(Zeff .- 1)) .+
        0.61 .* (1 .- 0.41 .* fT) .* νe_star ./ sqrt.(Zeff)
    )

    # Redl 2021, Eq. 17
    F33 = 1 .-
          (1 .+ 0.21 ./ Zeff) .* f33teff .+
          0.54 ./ Zeff .* f33teff.^2 .-
          0.33 ./ Zeff .* f33teff.^3

    σ_neo = σ_spitzer .* F33

    return σ_neo, σ_spitzer
end

In [ ]:
function calc_resistivity_P(chi, ne, Te, Zeff, q, eps, R, fT, volume;
                            model=:neoclassical)
    cond = neoclassical_conductivity(
        Te=Te,
        ne=ne,
        Ti=Te,
        Zeff=Zeff,
        q=q,
        eps=eps,
        R=R,
        fT=fT,
        volume=volume,
        version=:neo_2021,
    )

    σ = model == :neoclassical ?
        cond.neoclassical_conductivity :
        cond.spitzer_conductivity

    η = 1 ./ σ
    P = (4π * 1e-7) .* abs.(chi) ./ η

    return η, P
end

In [ ]:
σ_neo, σ_spitzer = neoclassical_conductivity(
    Te_eV,
    ne_sel,
    Te_eV,       # Ti = Te
    qsing,
    eps_sel,
    R_sel,
    fT_sel;
    Zeff=zeff_sel,
)

η_neo     = 1.0 ./ σ_neo
η_spitzer = 1.0 ./ σ_spitzer

P = (4π * 1e-7) .* abs.(chi_sel) ./ η_neo

In [ ]:
σ_neo, σ_spitzer = neoclassical_conductivity(
    106.211805100300,      # Te [eV]
    1.005827348013049e19,  # ne [m^-3]
    106.211805100300,      # Ti [eV]
    6.00000000000048,      # q
    0.771873939341347 / 1.74333594120074,  # eps = rs / R0
    1.74333594120074,      # R [m]
    1, #fT;                    # trapped fraction
    1.80427095488192, # Zeff
)

η_neo     = 1.0 ./ σ_neo
η_spitzer = 1.0 ./ σ_spitzer

chi_sel = 1

P = (4π * 1e-7) .* abs.(chi_sel) ./ η_neo

In [ ]:
using LinearAlgebra # or relevant module providing neoclassical_conductivity

# --- Set 1 Inputs (5 radial points) ---
Te_eV   = [718.04269217, 342.58020426, 188.39435197, 95.70267912, 44.04668565] # Te and Ti [eV]
ne_sel  = [1.98796958e19, 1.80371630e19, 1.55527262e19, 1.23649097e19, 8.97867646e18] # ne [m^-3]
zeff_sel = [1.65619282, 1.64427584, 1.58198407, 1.55990923, 1.68572562] # Zeff
qsing   = [2.0, 3.0, 4.0, 5.0, 6.0] # q
eps_sel = [0.22284171, 0.28956909, 0.32302672, 0.34432513, 0.35141583] # eps = r/R
R_sel   = [1.72151657, 1.70693349, 1.69832026, 1.69192129, 1.68947514] # R [m]
fT_sel  = [0.6127298, 0.66890766, 0.68553652, 0.68355803, 0.67414449] # Trapped fraction fT

# Reference values from Python NCLASS run (for validation)
py_neo     = [7683580.66632273, 3011088.07938057, 1644303.89021149, 782055.27991453, 274193.4323623]
py_spitzer = [16324797.88300323, 5666135.22162908, 2468477.31865855, 941040.7938515, 290205.76181926]

# --- Run Neoclassical Conductivity in Julia ---
# Broadcasting across all 5 radial positions:
σ_neo, σ_spitzer = neoclassical_conductivity.(
    Te_eV,    # Te [eV]
    ne_sel,   # ne [m^-3]
    Te_eV,    # Ti [eV] (assumed Ti = Te)
    qsing,    # q
    eps_sel,  # eps
    R_sel,    # R [m]
    fT_sel,   # trapped fraction fT
    zeff_sel  # Zeff
)

# Collect tuple outputs into a vector of tuples
results = [
    neoclassical_conductivity(
        Te_eV[i], ne_sel[i], Te_eV[i], qsing[i], 
        eps_sel[i], R_sel[i], fT_sel[i], zeff_sel[i]
    ) for i in 1:length(Te_eV)
]

# Extract σ_neo (1st element) and σ_spitzer (2nd element)
σ_neo     = getindex.(results, 1)
σ_spitzer = getindex.(results, 2)

η_neo     = 1.0 ./ σ_neo
η_spitzer = 1.0 ./ σ_spitzer

chi_sel = 1
P = (4π * 1e-7) .* abs.(chi_sel) ./ η_neo


# --- Comparison Printouts ---
println("=== Neoclassical Conductivity Comparison ===")
for i in 1:length(Te_eV)
    println("Point $i (q=$(qsing[i])):")
    println("  Julia σ_neo     : ", σ_neo[i])
    println("  Python σ_neo    : ", py_neo[i])
    println("  Julia σ_spitzer : ", σ_spitzer[i])
    println("  Python σ_spitzer: ", py_spitzer[i])
    println("  P (magnetic diff): ", P[i])
    println()
end